In [1]:
import pandas as pd
import re
import numpy as np
from tqdm import tqdm
import html
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize, sent_tokenize
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

nltk.download('stopwords')
nltk.download('punkt_tab')
nltk.download('punkt')

tqdm.pandas()

print("Libraries imported successfully")

Libraries imported successfully


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [2]:
df = pd.read_csv("rawData/dataNews.csv")
print(f"Data loaded. Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

# Pastikan kolom 'content' ada
if 'content' not in df.columns:
    # Coba cari kolom dengan nama berbeda
    text_columns = [col for col in df.columns if 'content' in col.lower() or 'text' in col.lower() or 'article' in col.lower()]
    if text_columns:
        df = df.rename(columns={text_columns[0]: 'content'})
        print(f"Renamed column '{text_columns[0]}' to 'content'")
    else:
        raise ValueError("No 'content' column found in dataset")

# Pilih kolom yang diperlukan
columns_to_keep = ['content']
additional_cols = ['title,publish_date']
for col in additional_cols:
    if col in df.columns:
        columns_to_keep.append(col)

df = df[columns_to_keep]
df['content'] = df['content'].fillna('').astype(str)

print(f"Data setelah seleksi kolom: {df.shape}")

Data loaded. Shape: (234, 3)
Columns: ['title', 'publish_date', 'content']
Data setelah seleksi kolom: (234, 1)


In [3]:
initial_count = len(df)
df.drop_duplicates(subset=['content'], inplace=True)
df.reset_index(drop=True, inplace=True)
print(f"Removed {initial_count - len(df)} duplicate articles")

# Tampilkan sampel
print("\n📄 SAMPLE CONTENT (first 500 chars):")
print("-" * 50)
print(df['content'].iloc[0][:500] + "...")
print("-" * 50)

Removed 0 duplicate articles

📄 SAMPLE CONTENT (first 500 chars):
--------------------------------------------------
Korps Pemberantasan Tindak Pidana Korupsi (Kortas Tipikor) Polri mengungkap kasus dugaan korupsi pada Ditjen Energi Baru Terbarukan dan Konservasi Energi (EBTKE) Kementerian ESDM terkait pengadaan penerangan jalan umum tenaga surya (PJUTS). Polri menyebut kerugian dalam kasus ini Rp 19.522.256.578,74.

Dirtindak Kortastipidkor Brigjen Totok Suharyanto mengatakan ada tiga orang yang ditetapkan sebagai tersangka dalam kasus tersebut. Ketiga tersangka ialah AS selaku mantan Inspektur Jenderal Kemen...
--------------------------------------------------


In [4]:
def clean_news_text(text):
    """
    Advanced cleaning khusus untuk teks berita
    - Menjaga singkatan resmi (KPK, Polri, dll)
    - Menangani angka dan mata uang dengan spesifik
    - Menghapus elemen non-berita
    """
    if not isinstance(text, str) or len(text.strip()) == 0:
        return ""
    
    # 1. Decode HTML entities
    text = html.unescape(text)
    
    # 2. Hapus bagian non-berita
    text = re.sub(r'SCROLL TO CONTINUE WITH CONTENT', '', text, flags=re.IGNORECASE)
    text = re.sub(r'ADVERTISEMENT', '', text, flags=re.IGNORECASE)
    text = re.sub(r'Baca juga.*', '', text, flags=re.IGNORECASE)
    text = re.sub(r'Simak juga.*', '', text, flags=re.IGNORECASE)
    text = re.sub(r'Video.*', '', text, flags=re.IGNORECASE)
    
    # 3. Normalisasi tanda kutip
    text = re.sub(r'""', '"', text)  # Ganti "" dengan "
    text = re.sub(r"''", "'", text)  # Ganti '' dengan '
    
    # 4. Hapus URL
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    
    # 5. CASE FOLDING (ke lowercase) - tapi jaga singkatan penting
    # Simpan singkatan penting sebelum case folding
    abbreviations = re.findall(r'\b[A-Z]{2,}\b', text)  # 2+ huruf kapital
    
    # Case folding
    text = text.lower()
    
    # 6. Tangani angka dan mata uang
    # Format: Rp 19.522.256.578,74 → rp1952225657874
    text = re.sub(r'rp\s*([\d.,]+)', lambda m: 'rp' + m.group(1).replace('.', '').replace(',', ''), text)
    
    # Format persentase: 25% → 25persen
    text = re.sub(r'(\d+)%', r'\1persen', text)
    
    # 7. Hapus karakter khusus tapi jaga beberapa untuk konteks
    # Jaga tanda kutip untuk kutipan langsung
    text = re.sub(r'[^\w\s"\'\.]', ' ', text)
    
    # 8. Tangani singkatan umum di berita
    abbreviation_mapping = {
        'kpk': 'komisi pemberantasan korupsi',
        'polri': 'kepolisian republik indonesia',
        'esdm': 'energi dan sumber daya mineral',
        'ditjen': 'direktorat jenderal',
        'ebtke': 'energi baru terbarukan dan konservasi energi',
        'pjuts': 'penerangan jalan umum tenaga surya',
        'ppk': 'pejabat pembuat komitmen',
        'kpa': 'kuasa pengguna anggaran',
        'uu': 'undang undang',
        'kuhp': 'kitab undang undang hukum pidana',
        'rp': 'rupiah',
        'pt': 'perseroan terbatas',
        'dll': 'dan lain lain',
    }
    
    for abbrev, expanded in abbreviation_mapping.items():
        text = re.sub(r'\b' + abbrev + r'\b', expanded, text)
    
    # 9. Hapus angka berdiri sendiri (tapi jaga yang ada konteks)
    # text = re.sub(r'\b\d+\b', '', text)  # Opsional: hapus angka
    
    # 10. Normalisasi whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

In [5]:
print("\n Applying cleaning to news articles...")
df['clean_content'] = df['content'].progress_apply(clean_news_text)
print("Cleaning completed")

# Tampilkan perbandingan
print("\n BEFORE vs AFTER CLEANING:")
print("-" * 50)
print("BEFORE (first 300 chars):")
print(df['content'].iloc[0][:300])
print("\nAFTER (first 300 chars):")
print(df['clean_content'].iloc[0][:300])
print("-" * 50)


 Applying cleaning to news articles...


100%|██████████| 234/234 [00:00<00:00, 983.08it/s] 

Cleaning completed

 BEFORE vs AFTER CLEANING:
--------------------------------------------------
BEFORE (first 300 chars):
Korps Pemberantasan Tindak Pidana Korupsi (Kortas Tipikor) Polri mengungkap kasus dugaan korupsi pada Ditjen Energi Baru Terbarukan dan Konservasi Energi (EBTKE) Kementerian ESDM terkait pengadaan penerangan jalan umum tenaga surya (PJUTS). Polri menyebut kerugian dalam kasus ini Rp 19.522.256.578,7

AFTER (first 300 chars):
korps pemberantasan tindak pidana korupsi kortas tipikor kepolisian republik indonesia mengungkap kasus dugaan korupsi pada direktorat jenderal energi baru terbarukan dan konservasi energi energi baru terbarukan dan konservasi energi kementerian energi dan sumber daya mineral terkait pengadaan pener
--------------------------------------------------


In [6]:
print("\n Loading normalization dictionary...")

# Kamus khusus untuk istilah berita/korupsi
news_normalization_dict = {
    # Istilah korupsi
    'korupsi': 'korupsi',
    'tipikor': 'tindak pidana korupsi',
    'tindak pidana': 'tindak pidana',
    'penyuapan': 'penyuapan',
    'gratifikasi': 'gratifikasi',
    'mark up': 'markup',
    'mark-up': 'markup',
    
    # Istilah hukum
    'tersangka': 'tersangka',
    'terdakwa': 'terdakwa',
    'saksi': 'saksi',
    'ahli': 'ahli',
    'penggeledahan': 'penggeledahan',
    'penyitaan': 'penyitaan',
    'blokir': 'pemblokiran',
    
    # Istilah lelang/pengadaan
    'lelang': 'pelelangan',
    'pengadaan': 'pengadaan',
    'pemaketan': 'pemaketan',
    'spesifikasi': 'spesifikasi',
    'review ulang': 'tinjauan ulang',
    'post-bidding': 'pasca pelelangan',
    
    # Singkatan umum
    'kortas': 'korps',
    'kortastipidkor': 'korps pemberantasan tindak pidana korupsi',
    'dirtindak': 'direktur tindak pidana',
    'jenderal': 'jenderal',
    'inspektur': 'inspektur',
    'direktur': 'direktur',
    'sekaligus': 'dan juga',
}

# Coba load kamus dari file jika ada
try:
    kamus_file = pd.read_excel("kamuskatabaku.xlsx")
    file_dict = dict(zip(kamus_file['tidak_baku'].str.lower(), kamus_file['kata_baku'].str.lower()))
    news_normalization_dict.update(file_dict)
    print(f"✅ Loaded {len(file_dict)} entries from kamuskatabaku.xlsx")
except Exception as e:
    print(f"⚠️  Could not load external dictionary: {e}")
    print("Using built-in dictionary only")

print(f"Total normalization entries: {len(news_normalization_dict)}")


 Loading normalization dictionary...
✅ Loaded 4347 entries from kamuskatabaku.xlsx
Total normalization entries: 4374


In [7]:
def normalize_news_text(text):
    """Normalize text with special handling for news terminology"""
    if not isinstance(text, str):
        return ""
    
    # Split into sentences first to preserve structure
    sentences = sent_tokenize(text)
    normalized_sentences = []
    
    for sentence in sentences:
        words = sentence.split()
        normalized_words = []
        
        for word in words:
            # Cek di kamus normalisasi
            if word in news_normalization_dict:
                normalized_words.append(news_normalization_dict[word])
            else:
                # Tangani kata berulang (misal: "sangat-sangat")
                word = re.sub(r'(\w+)-\1', r'\1', word)
                normalized_words.append(word)
        
        normalized_sentences.append(" ".join(normalized_words))
    
    return " ".join(normalized_sentences)

print("\n Applying normalization...")
df['normalized'] = df['clean_content'].progress_apply(normalize_news_text)
print(" Normalization completed")



 Applying normalization...


100%|██████████| 234/234 [00:00<00:00, 857.05it/s]

 Normalization completed


In [8]:
print("\n Tokenizing text...")

# Tokenize ke dalam kalimat
df['sentences'] = df['normalized'].progress_apply(
    lambda x: sent_tokenize(x) if isinstance(x, str) and len(x) > 0 else []
)

# Tokenize ke dalam kata
df['tokens'] = df['normalized'].progress_apply(
    lambda x: word_tokenize(x) if isinstance(x, str) and len(x) > 0 else []
)

print(f" Tokenization completed")
print(f"Average sentences per article: {df['sentences'].apply(len).mean():.1f}")
print(f"Average tokens per article: {df['tokens'].apply(len).mean():.1f}")



 Tokenizing text...


100%|██████████| 234/234 [00:00<00:00, 502.47it/s]

 Tokenization completed
Average sentences per article: 24.2
Average tokens per article: 451.0


In [9]:
print("\n Removing stopwords...")

# Stopwords dasar bahasa Indonesia
stop_words = set(stopwords.words('indonesian'))

# HAPUS stopwords yang mungkin penting dalam konteks berita
important_news_words = {
    'kpk', 'polri', 'korupsi', 'uang', 'rupiah', 'miliar', 'triliun',
    'tersangka', 'saksi', 'ahli', 'pasal', 'uu', 'hukum', 'pidana',
    'negara', 'kerugian', 'pengadaan', 'lelang', 'kontrak', 'proyek'
}

# Hanya hapus stopwords umum, jangan hapus kata penting
stop_words = {word for word in stop_words if word not in important_news_words}

# Tambahkan stopwords khusus berita yang kurang relevan
news_specific_stopwords = {
    'yang', 'dan', 'di', 'ke', 'dari', 'untuk', 'pada', 'dengan',
    'oleh', 'karena', 'atau', 'telah', 'akan', 'dalam', 'ini', 'itu',
    'juga', 'ada', 'adalah', 'merupakan', 'tersebut'
}
stop_words.update(news_specific_stopwords)

def remove_stopwords(tokens):
    return [w for w in tokens if w not in stop_words and len(w) > 2]

df['tokens_clean'] = df['tokens'].progress_apply(remove_stopwords)

print(f"✅ Stopword removal completed")
print(f"Tokens before: {df['tokens'].apply(len).mean():.1f}")
print(f"Tokens after: {df['tokens_clean'].apply(len).mean():.1f}")


 Removing stopwords...


100%|██████████| 234/234 [00:00<00:00, 22273.17it/s]

✅ Stopword removal completed
Tokens before: 451.0
Tokens after: 241.5


In [10]:
print("\n Applying stemming...")

factory = StemmerFactory()
stemmer = factory.create_stemmer()

def stem_news_tokens(tokens):
    stemmed = []
    
    # Daftar istilah yang TIDAK distem
    no_stem_words = {
        'korupsi', 'koruptor', 'polri', 'rupiah', 'miliar', 'triliun',
        'kpk', 'esdm', 'ebtke', 'pjuts', 'kuhp', 'undang', 'pasal',
        'tersangka', 'saksi', 'ahli', 'pidana', 'hukum'
    }
    
    for token in tokens:
        if token in no_stem_words or len(token) <= 3:
            stemmed.append(token)
        else:
            stemmed.append(stemmer.stem(token))
    
    return stemmed

df['stemmed'] = df['tokens_clean'].progress_apply(stem_news_tokens)
print(" Stemming completed")


 Applying stemming...


100%|██████████| 234/234 [02:45<00:00,  1.41it/s]

 Stemming completed


In [11]:
print("\n Creating final text...")

def create_final_text(stemmed_tokens):
    """Gabungkan tokens jadi teks dengan preservasi entitas"""
    return " ".join(stemmed_tokens) if stemmed_tokens else ""

df['final_text'] = df['stemmed'].apply(create_final_text)

# Buat versi dengan entitas penting di-tag
def tag_entities(text):
    """Tag entitas penting untuk analisis lebih lanjut"""
    entities = {
        'komisi pemberantasan korupsi': '[INST_KPK]',
        'kepolisian republik indonesia': '[INST_POLRI]',
        'energi dan sumber daya mineral': '[INST_ESDM]',
        'direktorat jenderal': '[INST_DITJEN]',
        'energi baru terbarukan dan konservasi energi': '[INST_EBTKE]',
        'penerangan jalan umum tenaga surya': '[PRODUK_PJUTS]',
        'rupiah': '[MATAUANG_IDR]',
        'tindak pidana korupsi': '[KEJAHATAN_KORUPSI]',
        'perseroan terbatas': '[BENTUKUSAHA_PT]',
    }
    
    for entity, tag in entities.items():
        text = text.replace(entity, tag)
    
    return text

df['tagged_text'] = df['final_text'].apply(tag_entities)

print(" Final text created")



 Creating final text...
 Final text created


In [12]:
print("\n" + "="*60)
print(" PREPROCESSING REPORT - NEWS DATA")
print("="*60)

# Statistik panjang
df['orig_length'] = df['content'].str.len()
df['final_length'] = df['final_text'].str.len()
df['orig_word_count'] = df['content'].apply(lambda x: len(str(x).split()))
df['final_word_count'] = df['final_text'].apply(lambda x: len(str(x).split()))

print(f"\n LENGTH STATISTICS:")
print(f"Original articles: {len(df)}")
print(f"Avg original length: {df['orig_length'].mean():.0f} chars")
print(f"Avg final length:    {df['final_length'].mean():.0f} chars")
print(f"Reduction:           {(1 - df['final_length'].mean()/df['orig_length'].mean())*100:.1f}%")
print(f"\nAvg original words:  {df['orig_word_count'].mean():.0f}")
print(f"Avg final words:     {df['final_word_count'].mean():.0f}")
print(f"Reduction:           {(1 - df['final_word_count'].mean()/df['orig_word_count'].mean())*100:.1f}%")

# Analisis kata paling umum
print(f"\n TOP 20 WORDS IN PROCESSED ARTICLES:")
all_tokens = []
for tokens in df['stemmed']:
    all_tokens.extend(tokens)

word_freq = pd.Series(all_tokens).value_counts().head(20)
for word, freq in word_freq.items():
    print(f"  {word:20s} : {freq:4d}")



 PREPROCESSING REPORT - NEWS DATA

 LENGTH STATISTICS:
Original articles: 234
Avg original length: 3042 chars
Avg final length:    1653 chars
Reduction:           45.7%

Avg original words:  417
Avg final words:     242
Reduction:           41.9%

 TOP 20 WORDS IN PROCESSED ARTICLES:
  etanol               : 1165
  energi               :  783
  bbm                  :  728
  bahan                :  702
  pertamina            :  616
  bakar                :  597
  indonesia            :  427
  campur               :  402
  sumber               :  379
  kandung              :  350
  daya                 :  349
  spbu                 :  346
  hasil                :  312
  bahlil               :  305
  2025                 :  286
  perintah             :  281
  bensin               :  280
  minyak               :  279
  mineral              :  273
  negara               :  272


In [13]:
print("\n" + "="*60)
print(" SAMPLE OUTPUT - COMPARISON")
print("="*60)

for i in range(min(2, len(df))):
    print(f"\n📌 ARTICLE {i+1}:")
    print(f"\nORIGINAL (first 400 chars):")
    print("-" * 40)
    print(df['content'].iloc[i][:400] + "...")
    
    print(f"\nFINAL PREPROCESSED (first 400 chars):")
    print("-" * 40)
    print(df['final_text'].iloc[i][:400] + "...")
    
    print(f"\nTAGGED VERSION (first 400 chars):")
    print("-" * 40)
    print(df['tagged_text'].iloc[i][:400] + "...")
    
    print(f"\nKEY ENTITIES:")
    print("-" * 40)
    tokens = df['stemmed'].iloc[i]
    unique_tokens = list(set(tokens))
    # Tampilkan kata unik yang panjangnya > 5
    long_tokens = [t for t in unique_tokens if len(t) > 5]
    print(", ".join(long_tokens[:15]))
    
    if i < 1:  # Hanya untuk artikel pertama
        print(f"\nPROCESSING PIPELINE (first 100 chars each):")
        print(f"Clean:      {df['clean_content'].iloc[i][:100]}...")
        print(f"Normalized: {df['normalized'].iloc[i][:100]}...")
        print(f"Tokens:     {df['tokens_clean'].iloc[i][:10]}...")
        print(f"Stemmed:    {df['stemmed'].iloc[i][:10]}...")


 SAMPLE OUTPUT - COMPARISON

📌 ARTICLE 1:

ORIGINAL (first 400 chars):
----------------------------------------
Korps Pemberantasan Tindak Pidana Korupsi (Kortas Tipikor) Polri mengungkap kasus dugaan korupsi pada Ditjen Energi Baru Terbarukan dan Konservasi Energi (EBTKE) Kementerian ESDM terkait pengadaan penerangan jalan umum tenaga surya (PJUTS). Polri menyebut kerugian dalam kasus ini Rp 19.522.256.578,74.

Dirtindak Kortastipidkor Brigjen Totok Suharyanto mengatakan ada tiga orang yang ditetapkan seba...

FINAL PREPROCESSED (first 400 chars):
----------------------------------------
korps berantas tindak pidana korupsi korps tindak pidana korupsi polisi republik indonesia ungkap duga korupsi direktorat jenderal energi baru konservasi energi energi baru konservasi energi menteri energi sumber daya mineral kait ada terang jalan tenaga surya terang jalan tenaga surya polisi republik indonesia sebut rugi rp1952225657874 direktur tindak pidana korps berantas tindak pidana korupsi ...

In [14]:
print("\n" + "="*60)
print(" EXPORTING RESULTS")
print("="*60)

# Simpan semua kolom
output_file = "cleanData/processed_news_data.csv"
df.to_csv(output_file, index=False, encoding='utf-8')
print(f" Full dataset saved to: {output_file}")


print(f"\n PREPROCESSING COMPLETE!")
print(f"Total news processed: {len(df)}")


 EXPORTING RESULTS
 Full dataset saved to: cleanData/processed_news_data.csv

 PREPROCESSING COMPLETE!
Total news processed: 234
